# Markovian GcStar Analysis - Notebook Simulation

This notebook runs a comprehensive analysis of the GcStar estimator across multiple simulated datasets, varying the conditioning depth (n_pasts) to understand how performance changes with different degrees of temporal conditioning.

## Objective
- Simulate 10 random datasets with varying network sizes
- Test GcStar with n_pasts ranging from 1 to 7
- Collect performance metrics (accuracy, precision, recall, F1, etc.)
- Visualize how metrics change with conditioning depth

## Configuration
- **Number of samples**: 10
- **Network size range**: 18-30 variables
- **Time series length**: 5000-10000 timesteps
- **Method**: c-GC (Conditional Granger Causality)
- **Permutations**: 1000

In [1]:
# Setup and imports

from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

# Import GcStar from causalised-GC
import importlib.util
import sys
CAUSALISED_GC_RELATIVE_PATH = Path("src/markovianity_diagnostic/core/causalised-GC.py")
PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / CAUSALISED_GC_RELATIVE_PATH).exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find '{CAUSALISED_GC_RELATIVE_PATH}' from {Path.cwd().resolve()}"
    )
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
CAUSALISED_GC_PATH = PROJECT_ROOT / CAUSALISED_GC_RELATIVE_PATH
spec = importlib.util.spec_from_file_location("causalised_gc", CAUSALISED_GC_PATH)
causalised_gc = importlib.util.module_from_spec(spec)
sys.modules["causalised_gc"] = causalised_gc
assert spec.loader is not None
spec.loader.exec_module(causalised_gc)
GcStar = causalised_gc.GcStar

from markovianity_diagnostic.core.utils import adj_mtx, continuous_noise_fun
from markovianity_diagnostic.experiments.graph_metrics import compute_graph_stability_metrics

print("✓ All imports successful")

✓ All imports successful


In [2]:
# Configuration parameters
NUM_TRIALS = 10
N_PASTS = list(range(1, 8))  # Test n_pasts from 1 to 7
ALPHA = 0.01
BETA = 0.001
N_LAGS = 1
N_PERM = 1000
GROUND_TRUTH_AVAILABLE = True

print("Configuration:")
print(f"  Trials: {NUM_TRIALS}")
print(f"  n_pasts: {N_PASTS}")
print(f"  Alpha: {ALPHA}, Beta: {BETA}")
print(f"  Permutations: {N_PERM}")
print(f"  Ground truth available: {GROUND_TRUTH_AVAILABLE}")

Configuration:
  Trials: 10
  n_pasts: [1, 2, 3, 4, 5, 6, 7]
  Alpha: 0.01, Beta: 0.001
  Permutations: 1000
  Ground truth available: True


## Step 1: Run Simulations

The cell below simulates 10 random datasets and tests GcStar with each n_pasts value. This may take a few minutes.

In [ ]:
# Initialize results storage
all_results = {
    'config': {
        'num_samples': NUM_TRIALS,
        'n_pasts': N_PASTS,
        'alpha': ALPHA,
        'beta': BETA,
        'n_lags': N_LAGS,
        'n_perm': N_PERM,
        'ground_truth_available': GROUND_TRUTH_AVAILABLE,
    },
    'samples': []
}

print("Starting simulations...")
print("=" * 80)

# Run simulations
for sample_idx in tqdm(range(NUM_TRIALS), desc="Samples"):
    # Random network configuration
    n_neur = np.random.randint(18, 30, 1)[0]
    l = np.random.randint(5000, 10000, 1)[0]

    # Simulate data
    A = adj_mtx(n_neur)
    X = np.zeros((A.shape[0], l)).T
    X[0] = np.random.randn(A.shape[0])
    for i, row in enumerate(X[:-1]):
        X[i+1] = A @ X[i] + np.random.normal(0, 0.25, A.shape[0])
    X = X.T

    adjacencies = {}
    metrics_by_n_past = {}
    shd_by_n_past = {}

    for n_past in N_PASTS:
        gcstar = GcStar(
            n_perm=N_PERM,
            n_pasts=n_past,
            n_lags=N_LAGS,
            temporal=True,
            method="cgc"
        )
        gcstar.fit(X, verbose=0)
        conn_mat = gcstar.get_connectivity_matrix(alpha=ALPHA, beta=BETA, simulation=True)
        adjacencies[n_past] = conn_mat

        if GROUND_TRUTH_AVAILABLE:
            gcstar.compute_confusion_matrix(A, simulation=True)
            metrics = gcstar.compute_metrics()
            metrics_by_n_past[n_past] = {
                'accuracy': float(np.round(metrics[0], 4)),
                'precision': float(np.round(metrics[1], 4)),
                'recall': float(np.round(metrics[2], 4)),
                'fpr': float(np.round(metrics[3], 4)),
                'balanced_accuracy': float(np.round(metrics[4], 4)),
                'f1': float(np.round(metrics[5], 4)),
            }
            shd_by_n_past[n_past] = float(gcstar.compute_shd_sid(A, conn_mat, simulation=True))

    if GROUND_TRUTH_AVAILABLE:
        sample_metrics = {
            'sample_idx': sample_idx,
            'n_neur': int(n_neur),
            'n_timesteps': int(l),
            'metrics_by_n_past': metrics_by_n_past,
            'shd_by_n_past': shd_by_n_past,
        }
    else:
        stability = compute_graph_stability_metrics(adjacencies)
        sample_metrics = {
            'sample_idx': sample_idx,
            'n_neur': int(n_neur),
            'n_timesteps': int(l),
            'T_obs': stability['T_obs'],
            'edge_counts': stability['edge_counts'],
            'D_p': stability['D_p'],
            'D_parts': stability['D_parts'],
        }

    all_results['samples'].append(sample_metrics)

print("\n✓ Simulations complete!")
print(f"Processed {NUM_TRIALS} samples × {len(N_PASTS)} n_pasts values")

Starting simulations...


Samples:   0%|          | 0/10 [00:00<?, ?it/s]

## Step 2: Inspect Individual Sample Results

View the raw results from all samples to understand the structure and individual performance.

When `GROUND_TRUTH_AVAILABLE = True`, the notebook stores recovery metrics from `causalised-GC.py`:
- `metrics_by_n_past`: accuracy, precision, recall, fpr, balanced_accuracy, f1
- `shd_by_n_past`: SHD per conditioning depth

When `GROUND_TRUTH_AVAILABLE = False`, the notebook stores stability metrics instead:
- `T_obs`
- `edge_counts`
- `D_p`
- `D_parts`

In [ ]:
# Show first sample in detail
print("Sample 0 Results:")
print("=" * 80)
sample_0 = all_results['samples'][0]
print(f"Network size: {sample_0['n_neur']} variables")
print(f"Time series length: {sample_0['n_timesteps']} timesteps")
print("\nMetrics by n_past:")
print(json.dumps(sample_0['metrics_by_n_past'], indent=2))

## Step 3: Compute Aggregated Statistics

Calculate mean and standard deviation across all samples for each n_pasts value.

In [ ]:
# Organize metrics by n_past
metrics_by_npast = {str(p): [] for p in N_PASTS}
for sample in all_results['samples']:
    for n_past_str, metrics_dict in sample['metrics_by_n_past'].items():
        metrics_by_npast[n_past_str].append(metrics_dict)

# Compute mean and std for each metric at each n_past
aggregated = {}
for n_past_str in [str(p) for p in N_PASTS]:
    metrics_list = metrics_by_npast[n_past_str]
    
    aggregated[n_past_str] = {
        'accuracy': {
            'mean': float(np.round(np.mean([m['accuracy'] for m in metrics_list]), 4)),
            'std': float(np.round(np.std([m['accuracy'] for m in metrics_list]), 4))
        },
        'precision': {
            'mean': float(np.round(np.mean([m['precision'] for m in metrics_list]), 4)),
            'std': float(np.round(np.std([m['precision'] for m in metrics_list]), 4))
        },
        'recall': {
            'mean': float(np.round(np.mean([m['recall'] for m in metrics_list]), 4)),
            'std': float(np.round(np.std([m['recall'] for m in metrics_list]), 4))
        },
        'fpr': {
            'mean': float(np.round(np.mean([m['fpr'] for m in metrics_list]), 4)),
            'std': float(np.round(np.std([m['fpr'] for m in metrics_list]), 4))
        },
        'balanced_accuracy': {
            'mean': float(np.round(np.mean([m['balanced_accuracy'] for m in metrics_list]), 4)),
            'std': float(np.round(np.std([m['balanced_accuracy'] for m in metrics_list]), 4))
        },
        'f1': {
            'mean': float(np.round(np.mean([m['f1'] for m in metrics_list]), 4)),
            'std': float(np.round(np.std([m['f1'] for m in metrics_list]), 4))
        }
    }

all_results['aggregated_statistics'] = aggregated

print("✓ Aggregated statistics computed")
print(f"Data shape: {NUM_SAMPLES} samples × {len(N_PASTS)} n_pasts values")

In [ ]:
# Display aggregated statistics as a table
print("\nAggregated Results: Mean ± Std Across Samples")
print("=" * 100)
print(f"{'n_past':<10} {'Accuracy':<18} {'Precision':<18} {'Recall':<18} {'F1':<18}")
print("-" * 100)
for n_past in N_PASTS:
    agg = aggregated[str(n_past)]
    acc = f"{agg['accuracy']['mean']:.4f}±{agg['accuracy']['std']:.4f}"
    prec = f"{agg['precision']['mean']:.4f}±{agg['precision']['std']:.4f}"
    rec = f"{agg['recall']['mean']:.4f}±{agg['recall']['std']:.4f}"
    f1 = f"{agg['f1']['mean']:.4f}±{agg['f1']['std']:.4f}"
    print(f"{n_past:<10} {acc:<18} {prec:<18} {rec:<18} {f1:<18}")

## Step 4: Save Results to File

Store the complete results for later analysis and sharing.

In [ ]:
# Save results
output_file = 'notebook_simulation_results.json'
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"✓ Results saved to '{output_file}'")
print(f"File size: {Path(output_file).stat().st_size / 1024:.1f} KB")

## Step 5: Visualize Results

Create line plots showing how each metric changes with n_pasts.

In [ ]:
# Prepare data for visualization
metric_names = ['accuracy', 'precision', 'recall', 'fpr', 'balanced_accuracy', 'f1']
n_pasts_range = [str(p) for p in N_PASTS]

# Extract means and stds
data_by_metric = {}
for metric in metric_names:
    means = [aggregated[np_str][metric]['mean'] for np_str in n_pasts_range]
    stds = [aggregated[np_str][metric]['std'] for np_str in n_pasts_range]
    data_by_metric[metric] = {'mean': means, 'std': stds}

# Create visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('GcStar Performance Metrics vs n_pasts (Conditioning Depth)', fontsize=16, fontweight='bold')

metric_labels = {
    'accuracy': 'Accuracy',
    'precision': 'Precision',
    'recall': 'Recall',
    'fpr': 'False Positive Rate',
    'balanced_accuracy': 'Balanced Accuracy',
    'f1': 'F1 Score'
}

axes_flat = axes.flatten()
for idx, metric in enumerate(metric_names):
    ax = axes_flat[idx]
    means = data_by_metric[metric]['mean']
    stds = data_by_metric[metric]['std']

    ax.plot(N_PASTS, means, 'o-', linewidth=2, markersize=8, label='Mean')
    ax.fill_between(N_PASTS,
                    np.array(means) - np.array(stds),
                    np.array(means) + np.array(stds),
                    alpha=0.3, label='±1 Std Dev')

    ax.set_xlabel('n_pasts (Conditioning Depth)', fontsize=11)
    ax.set_ylabel(metric_labels[metric], fontsize=11)
    ax.set_title(metric_labels[metric], fontsize=12, fontweight='bold')
    ax.set_xticks(N_PASTS)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

    # Set appropriate y-axis limits
    if metric == 'fpr':
        ax.set_ylim([0, 0.3])
    else:
        ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('gcstar_metrics_by_npasts.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualization saved as 'gcstar_metrics_by_npasts.png'")

## Step 6: Analysis Summary

Key observations and insights from the results.

In [ ]:
# Summary statistics
print("Analysis Summary")
print("=" * 80)

print("\n1. ACCURACY TREND:")
acc_means = [aggregated[str(p)]['accuracy']['mean'] for p in N_PASTS]
print(f"   Range: {min(acc_means):.4f} - {max(acc_means):.4f}")
print(f"   Best at n_past={N_PASTS[np.argmax(acc_means)]} (acc={max(acc_means):.4f})")
print(f"   Worst at n_past={N_PASTS[np.argmin(acc_means)]} (acc={min(acc_means):.4f})")

print("\n2. RECALL TREND (True Positive Rate):")
rec_means = [aggregated[str(p)]['recall']['mean'] for p in N_PASTS]
print(f"   Range: {min(rec_means):.4f} - {max(rec_means):.4f}")
print(f"   Best at n_past={N_PASTS[np.argmax(rec_means)]} (recall={max(rec_means):.4f})")

print("\n3. FPR TREND (False Positive Rate):")
fpr_means = [aggregated[str(p)]['fpr']['mean'] for p in N_PASTS]
print(f"   Range: {min(fpr_means):.4f} - {max(fpr_means):.4f}")
print(f"   Best (lowest) at n_past={N_PASTS[np.argmin(fpr_means)]} (fpr={min(fpr_means):.4f})")

print("\n4. F1 SCORE TREND:")
f1_means = [aggregated[str(p)]['f1']['mean'] for p in N_PASTS]
print(f"   Range: {min(f1_means):.4f} - {max(f1_means):.4f}")
print(f"   Best at n_past={N_PASTS[np.argmax(f1_means)]} (f1={max(f1_means):.4f})")

print("\n5. SAMPLE VARIABILITY:")
for metric in ['accuracy', 'f1']:
    stds = [aggregated[str(p)][metric]['std'] for p in N_PASTS]
    avg_std = np.mean(stds)
    print(f"   {metric.capitalize()}: avg std={avg_std:.4f}")

## Step 7: Export for Further Analysis

Create a pandas DataFrame for easy manipulation and export.

In [ ]:
# Create DataFrame from aggregated results
rows = []
for n_past in N_PASTS:
    agg = aggregated[str(n_past)]
    row = {
        'n_past': n_past,
        'accuracy_mean': agg['accuracy']['mean'],
        'accuracy_std': agg['accuracy']['std'],
        'precision_mean': agg['precision']['mean'],
        'precision_std': agg['precision']['std'],
        'recall_mean': agg['recall']['mean'],
        'recall_std': agg['recall']['std'],
        'fpr_mean': agg['fpr']['mean'],
        'fpr_std': agg['fpr']['std'],
        'balanced_accuracy_mean': agg['balanced_accuracy']['mean'],
        'balanced_accuracy_std': agg['balanced_accuracy']['std'],
        'f1_mean': agg['f1']['mean'],
        'f1_std': agg['f1']['std'],
    }
    rows.append(row)

df = pd.DataFrame(rows)

# Save to CSV
csv_file = 'gcstar_results_summary.csv'
df.to_csv(csv_file, index=False)

print(f"✓ Results exported to '{csv_file}'")
print("\nDataFrame Preview:")
print(df.to_string())

## Next Steps

The results have been saved in multiple formats:
- **JSON**: `notebook_simulation_results.json` (detailed, per-sample results)
- **CSV**: `gcstar_results_summary.csv` (aggregated statistics)
- **PNG**: `gcstar_metrics_by_npasts.png` (visualization)

You can now:
1. Compare these results with the notebook's direct analysis
2. Explore different configurations (different n_perm, alpha, beta values)
3. Test with fcGC method instead of cgc
4. Export results for publication